# CS448 - Lab 10: Keyword Recognition

## Part 1: Making a DTW-based Digit Recognizer

In this section we will design a simple spoken digit recognizer, based on Dynamic Time Warping (DTW). In order to make such a system we need to first collect some data, and then design a DTW routine that can compare new inputs with templates for each digit.

There is a dataset called Free Spoken Digit Dataset (https://github.com/jakobovski/free-spoken-digit-dataset), where you can find a bunch of recordings of spoken digits. You can train a proper model by using all of them, but let's just focus on the small subset that's just good enough to prove the concept. Choose from ```*_george_0.wav``` to ```*_george_10.wav```, which will give you 11 utterances per digit, spoken by George. We will use ```*_george_0.wav``` as templates and the rest as test samples. For example, ```7_george_0``` is the template for the digit 7 classs, while all others start from 7 are test samples. 

If you are curious and want to engage more with the lab, you can record your own voices and create your own set, too. 

In order to design a digit recognizer we will take a spoken input of a digit and compare it to each digit’s template. By finding which template is the most similar we can classify the input as belonging to that template’s digit. In order to measure the distance between the two sequences we have to use DTW on an appropriate feature space.

Decide which feature to use to represent your speech signals. It can be any feature that we used in the past (e.g. some type of an STFT, MFCCs, etc). When comparing a template with a new input you need to perform the following steps:

1. Compute the distance matrix between all the features of each input. This will be a $M$ by $N$ matrix in which the $(i, j)$ element will represent the distance between the $i$-th frame of the template and the $j$-th frame of the input. We will use the cosine similarity which is defined as:

$$D(\mathbf{a},\mathbf{b}) = 1 - \frac{\sum a_i b_i}{\sqrt{\sum a_i^2}\sqrt{\sum b_i^2}}$$

2. Once you obtain the distance matrix, you need to compute the cost matrix that encodes the cost of passing through a node given a previously optimal path. We will use the local constraint that to reach node $(i, j)$ you can either come from nodes $(i–1, j–1)$, $(i, j–1)$ or $(i–1, j)$.

3. Starting from the first element of the matrix (0,0), and for each element of the cost matrix you will need to perform the following steps. For node $(i, j)$ you need to examine the nodes from which you can reach it – these will be nodes $(i–1, j–1)$, $(i, j–1)$ or $(i–1, j)$ – and see which one has the lowest cost. Therefore, reaching that node from the optimal path will have the cost of the optimal preceding node plus the distance that corresponds to being at node $(i, j)$. Iterate until you calculate the cost of passing through every node. As you do that, for each node keep track of which of the three preceding nodes was the optimal one. Let's not worry about the global constraint for this problem. 

4. Now you can backtrack and find the optimal path. Start from the final point of the cost matrix and find the node from which you arrived there (it will be the same one that had the lowest cost above). Once you get to that node, repeat this process until you reach the beginning indexes of the two sequences. The path that you took in this process will be the optimal path that aligns the two sequences.  Note that the final point will not necessarily be the far corner of the matrix, as the two sequences might align differently.  Look for the minimum value in the last row/column of the cost matrix and start backtracking from there.  Obviously, if the smallest cost is too close to the beginning, something is wrong (or you have a really bad match).

5. The distance between the two sequences will be the cost of being at the final node. Use this to perform the digit classification.


Draw a confusion matrix and see if you like the result.

In [1]:
import os
import requests

# Folder to save files
save_dir = "fsdd_george"
os.makedirs(save_dir, exist_ok=True)

base_url = (
    "https://raw.githubusercontent.com/"
    "Jakobovski/free-spoken-digit-dataset/master/recordings"
)

# Download files
for digit in range(10):
    for sample_num in range(11):
        filename = f"{digit}_george_{sample_num}.wav"
        url = f"{base_url}/{filename}"

        print(f"Downloading {filename}...")

        response = requests.get(url)

        if response.status_code == 200:
            filepath = os.path.join(save_dir, filename)

            with open(filepath, "wb") as f:
                f.write(response.content)

        else:
            print(f"Failed to download {filename}")

print("Done.")

Done.


In [ ]:
import librosa
import numpy as np

templates = []
for i in range(10):
    template, sr = librosa.load(f'data/{i}_george_0.wav')
    templates.append(template)


## Part 2. Making an HMM-based Digit Recognizer

We will now redo this experiment using HMMs. One issue with the previous setup is that it didn't work well for the 0 class, probably because ```0_george_0.wav``` not a good representative of George's way of saying zero. 

As a remedy, let's increase the number of templates to five. Instead of using all five of them for the DTW-style matching, we will learn a single HMM model per digit class. In other words, the HMM training function will take five spectrograms concatenated along the time direction as its input: this is far better than having five templates and running the DTW algorithm five times for each class. 

To do that you will need to install the ```hmmlearn``` package. I wouldn't ask you to implement the HMM training algorithm, because it's painful to do so, and impossible to do it within the lab session. 

We can use the same features as before, but we should make sure that they are somewhat Gaussian distributed so that the HMM assumptions are more valid. So, if you've used STFT magnitudes for the previous problem, let's switch to MFCC spectrograms. 

For each digit class, let's say ```X``` holds all the five MFCC spectrograms concatenated. You also want to keep a list of ```lengths``` because your HMM model needs it. For example, if you concatenate five matrices of sizes 30 x 13, 30 x 15, 30 x 20, 30 x 10, and 30 x 12, then your ```X``` will be of size 30 x 70, while your ```lengths``` list should host the length information: ```[13, 15, 20,10, 12]```. 

You first initiate the ```hmmlearn.hmm.GaissuanHMM``` class with necessary arguments, such as ```n_components``` for the number of states (something roughly corresponds to the number of phonemes in that digit), ```covariance_time``` (just use ```'diag'``` for convenience), etc. Important ones are ```init_params``` which should be set with ```'mc'``` and ```params='mc'```. 

```init_params='mc'``` means that you initialize the underlying Gaussian parameters (the likelihood model of observing the MFCC vector given a hidden state) randomly, but you DON'T want to randomly initialize the hidden states-related parameters. It's because you want to initialize them manually and stick to them (you don't update them). For five hidden states, for example, 
```
model[digit_class].startprob_ = np.array([1.0, 0.0, 0.0, 0.0, 0.0])
model[digit_class].transmat_ = np.array([
        [0.95, 0.05, 0.0, 0.0, 0.0],
        [0.0, 0.95, 0.05, 0.0, 0.0],
        [0.0, 0.0, 0.95, 0.05, 0.0],
        [0.0, 0.0, 0.0, 0.95, 0.05],
        [0.0, 0.0, 0.0, 0.0, 1.]
    ]) 
```
these initialization makes sure that the forward pass starts from the 0-th hidden state, while the transition is regularized nicely, i.e., either to stay within the same state with 0.95 probability or jump to the next state with 5% of chance. With ```params='mc'```, your manual initialization won't be messed up during training. You only update the Gaussian parameters for likelihoods instead. Otherwise, the model parameters are too flexible with a lot of degree of freedom, making training unstable. This kind of regularization cane result in a *left-to-right* HMM. 

Then, use the ```hmmlearn.hmm.GaussianHMM.fit()``` function to train the model by feeding ```X``` and ```lengths``` to it. Note that the ```fit()``` function prefers ```T x F``` rather than ```F x T``` requiring a transposed version of ```X```. 

In addition, ```hmmlearn.hmm.GaussianHMM.score()``` function can calculate the matching score between the model and any test sequence. You want to run 10 HMM models for any given test sample to identify which class it belongs to, i.e., the model that gives you the highest score for that test sample represents the best-matching digit class. 

After prediction you can calculate the confusion matrix. Is it any better?

In [84]:
# YOUR CODES HERE